In [ ]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

# 06 – Semantic Search Demo

Interactive semantic search on the Reuters documents using a MiniLM embedding model. Optionally, this notebook can switch to a Retrieval‑Augmented Generation (RAG) mode that synthesises an answer from the top retrieved documents.

In [ ]:

import json, pathlib, faiss
from sentence_transformers import SentenceTransformer
from src.datasets.dataset import get_dataset

from src.rag import _SBERT_DIR, _EMBEDDINGS_DIR

# Use the correct embeddings directory from the configuration
INDEX_PATH = _SBERT_DIR / 'index.faiss'
DOCS_PATH = _SBERT_DIR / 'meta.jsonl'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'


In [ ]:
# Load documents (train + test for demo)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
)

In [ ]:
# Combine train and test documents
docs = X_train + X_test
labels = y_train + y_test

if not INDEX_PATH.exists():
    print('Building embeddings…')
    model = SentenceTransformer(MODEL_NAME)
    emb = model.encode(docs, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
    dimension = emb.shape[1]
    index = faiss.IndexFlatIP(dimension)
    # normalise for cosine sim
    faiss.normalize_L2(emb)
    index.add(emb)
    
    # Create metadata for each document
    meta = []
    for i, (txt, label) in enumerate(zip(docs, labels)):
        meta.append({
            "id": i,
            "text": txt,
            "label": label,
            "vector": emb[i].tolist()  # Store the vector in the metadata
        })
    
    # Ensure directory exists
    INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    # Save FAISS index
    faiss.write_index(index, str(INDEX_PATH))
    
    # Save metadata as JSONL
    with open(DOCS_PATH, 'w') as f:
        for m in meta:
            f.write(json.dumps(m) + '\n')
            
    print(f'Index and docs saved to {_SBERT_DIR}')
else:
    print('Embeddings already built. Loading…')
    index = faiss.read_index(str(INDEX_PATH))
    # Load metadata
    meta = []
    with open(DOCS_PATH) as f:
        for line in f:
            meta.append(json.loads(line))
    docs = [m['text'] for m in meta]
    model = SentenceTransformer(MODEL_NAME)


In [5]:
# Function to search for similar documents
def search(query: str, k: int = 5):
    # Encode query
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    
    # Search
    D, I = index.search(q_emb, k)
    
    # Return results
    results = []
    for score, idx in zip(D[0], I[0]):
        results.append({
            'text': meta[idx]['text'],
            'label': meta[idx]['label'],
            'score': float(score)
        })
    return results

In [ ]:

# Demo
search("oil prices in saudi arabia")


### Optional – Retrieval‑Augmented Generation (RAG)
If you have an OpenAI key configured, you can uncomment the cell below to generate answers from the retrieved passages.

In [3]:
# Install required packages
# !pip install openai

import os
import sys
import json
import pathlib
import textwrap
import numpy as np
from typing import List, Dict, Any

import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from pathlib import Path

# Setup paths and import project modules
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # Allow `import src.*`

from config.notebook_setup import *
from src.datasets.dataset import get_dataset
from src.rag import _SBERT_DIR, _EMBEDDINGS_DIR

# Configuration
INDEX_PATH = _SBERT_DIR / 'index.faiss'
DOCS_PATH = _SBERT_DIR / 'meta.jsonl'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

# Load dataset
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
    cutoff_year=GENERAL_CUTOFF_YEAR
)

docs = X_train + X_test
labels = y_train + y_test

# Build or load FAISS index
if not INDEX_PATH.exists():
    print('Building embeddings…')
    model = SentenceTransformer(MODEL_NAME)
    emb = model.encode(docs, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
    faiss.normalize_L2(emb)
    
    dimension = emb.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(emb)
    
    # Create and save metadata
    meta = [{"id": i, "text": txt, "label": label, "vector": emb[i].tolist()}
            for i, (txt, label) in enumerate(zip(docs, labels))]

    INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(INDEX_PATH))

    with open(DOCS_PATH, 'w') as f:
        for m in meta:
            f.write(json.dumps(m) + '\n')

    print(f'Index and docs saved to {_SBERT_DIR}')
else:
    print('Embeddings already built. Loading…')
    index = faiss.read_index(str(INDEX_PATH))
    with open(DOCS_PATH) as f:
        meta = [json.loads(line) for line in f]
    docs = [m['text'] for m in meta]
    model = SentenceTransformer(MODEL_NAME)

# Initialize OpenAI client
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY environment variable not set.')
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def analyze_document_similarity(query: str, doc: str, query_emb: np.ndarray, doc_emb: np.ndarray) -> Dict[str, Any]:
    """Analyze semantic similarity between a query and a document."""
    similarity = float(np.dot(query_emb, doc_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(doc_emb)))

    prompt = f"""Analyze the semantic relationship between this query and document:

Query: {query}

Document: {doc}

Provide a brief analysis of:
1. How relevant the document is to the query
2. Key topics or concepts that match
3. Any important information gaps

Analysis:"""

    response = client.completions.create(
        model='gpt-3.5-turbo-instruct',
        prompt=prompt,
        max_tokens=150
    )
    analysis = textwrap.dedent(response.choices[0].text).strip()

    return {
        'similarity_score': similarity,
        'semantic_analysis': analysis
    }

def rag_answer(question: str, k: int = 5) -> str:
    """Enhanced RAG implementation with similarity and semantic analysis."""
    q_emb = model.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    D, I = index.search(q_emb, k)

    references = []
    document_analyses = []

    for rank, (idx, score) in enumerate(zip(I[0], D[0]), 1):
        doc_text = docs[idx]
        doc_emb = np.array(meta[idx]['vector'])

        analysis = analyze_document_similarity(question, doc_text, q_emb[0], doc_emb)

        references.append(f"Document {rank} (score={score:.3f}): {doc_text}")
        document_analyses.append({
            'rank': rank,
            'similarity_score': analysis['similarity_score'],
            'semantic_analysis': analysis['semantic_analysis']
        })

    context = "\n".join([docs[i] for i in I[0]])
    prompt = f"""Answer the question based only on the context below.

Context:
{context}

Question: {question}

Provide a comprehensive answer that:
1. Directly addresses the question
2. Cites specific information from the reference documents
3. Highlights any uncertainties or information gaps

Answer:"""

    response = client.completions.create(
        model='gpt-3.5-turbo-instruct',
        prompt=prompt,
        max_tokens=512
    )
    answer = textwrap.dedent(response.choices[0].text).strip()

    analysis_text = "\n\nDocument Analysis:\n"
    for analysis in document_analyses:
        analysis_text += f"\nDocument {analysis['rank']}:\n"
        analysis_text += f"Similarity Score: {analysis['similarity_score']:.3f}\n"
        analysis_text += f"Analysis: {analysis['semantic_analysis']}\n"

    references_text = "\n\n".join(references)

    return f"""Answer:
{answer}

{analysis_text}

Reference Documents:
{references_text}"""

# Example usage
if __name__ == "__main__":
    question = "What were the effects of the Reagan administration on oil prices and energy policy?"
    print(rag_answer(question))


/home/marcmaceira/projects/reuters-rag-classifier_clean_v2/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2
Configuration: {'general': {'run_name': 'experiment_with_4_classes', 'seed': 42, 'n_classes': 4}, 'dataset': {'split_type': 'test'}, 'paths': {'data_exploration_dir': 'output/experiment_with_4_classes/data_exploration', 'embeddings_dir': 'output/experiment_with_4_classes/embeddings', 'models_dir': 'output/experiment_with_4_classes/models', 'predictions_dir': 'output/experiment_with_4_classes/predictions', 'results_dir': 'output/experiment_with_4_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_SPLIT_TYPE: test

[EVALUATION]
  EVALUATIO

TypeError: get_dataset() got an unexpected keyword argument 'cutoff_year'